# SMR multilink modelization


Combined Heat and Power (CHP) with fixed heat-power ratio. 

This [example](https://pypsa.readthedocs.io/en/latest/examples/chp-fixed-heat-power-ratio.html) demonstrates a Link component with more than one bus output (“bus2” in this case). In general links can have many output buses.

In this example a CHP must be heat-following because there is no other supply of heat to the bus “elec_Industry heat”.

In [4]:
import pypsa

In [ ]:
n0 = pypsa.Network() # PyPSA network object called n0

n0.add("Bus", "Industry Electricity", carrier="AC")
n0.add("Load", "Industry Electricity Load", bus="Industry Electricity", p_set=5)


n0.add("Bus", "Industry Heat", carrier="Heat")
n0.add("Load", "Industry Heat Load", bus="Industry Heat", p_set=3)


n0.add("Bus", "Nuclear_Fuel", carrier="Uranium")
n0.add("Store", "Nuclear_Fuel", 
       e_initial=2e6, 
       e_nom=1e6, 
       bus="Nuclear Fuel")

n1.add("Bus", "Spain gas", carrier="gas")
n1.add("Store", "Spain gas", 
       e_initial=1e6,   # Energy before the snapshots in the OPF (Optimized power Flow).
       e_nom=1e6,       # Nominal energy capacity
       bus="Spain gas")

n0.add(
    "Link",
    "OCGT",                 # Open Cycle Gas Turbine
    bus0="Spain gas",
    bus1="Industry Electricity",       # Only produces electricity
    p_nom_extendable=True,
    capital_cost=600,
    efficiency=0.4,
)

n0.add(
    "Link",
    "SMR_CHP",
    bus0="Nuclear_Fuel",
    bus1="Industry Electricity",
    bus2="Industry Heat",
    p_nom_extendable=True,
    capital_cost=1400,
    efficiency=0.3,
    efficiency2=0.3,
)

Index(['CHP'], dtype='object')

## About de network initialization: 

2. **Network Initialization**: `n1 = pypsa.Network(url)`
   - Creates a PyPSA `Network` object `n1`
   - Automatically downloads and loads the pre-configured energy system model containing:
     - Buses (energy nodes)
     - Generators (power plants)
     - Loads (energy demands)
     - Transmission lines
     - Time-dependent constraints

### Key Features of This Approach:
- **Reproducibility**: Loads a standardized benchmark model
- **Time-Saving**: Avoids manual network construction (~1000+ lines of setup code)
- **Research-Ready**: Contains realistic European power system data (likely from [PyPSA-Eur](https://pypsa-eur.readthedocs.io/))

In [ ]:
url = "https://tubcloud.tu-berlin.de/s/pzytNg9gtkgPpXc/download/network-cem.nc" # specify PyPSA network file hosted on TU Berlin´s cloud
n1 = pypsa.Network() # PyPSA network object called n1 (automatically downloads and loads the pre-configured energy system model)

n1.add("Bus", "Frankfurt", carrier="AC")
n1.add("Load", "Frankfurt", bus="Frankfurt", p_set=5)


n1.add("Bus", "Frankfurt heat", carrier="heat")
n1.add("Load", "Frankfurt heat", bus="Frankfurt heat", p_set=3)


n1.add("Bus", "Frankfurt gas", carrier="gas")
n1.add("Store", "Frankfurt gas", 
       e_initial=1e6,   # Energy before the snapshots in the OPF (Optimized power Flow).
       e_nom=1e6,       # Nominal energy capacity
       bus="Frankfurt gas")

n1.add(
    "Link",
    "OCGT",                 # Open Cycle Gas Turbine
    bus0="Frankfurt gas",
    bus1="Frankfurt",       # the output only produces electricity
    p_nom_extendable=True,
    capital_cost=600,
    efficiency=0.4,
)


n1.add(
    "Link",
    "CHP",
    bus0="Frankfurt gas",
    bus1="Frankfurt",
    bus2="Frankfurt heat",
    p_nom_extendable=True,
    capital_cost=1400,
    efficiency=0.3,
    efficiency2=0.3,
)

Index(['CHP'], dtype='object')

In [10]:
n1

Unnamed PyPSA Network
---------------------
Components:
 - Bus: 3
 - Link: 2
 - Load: 2
 - Store: 1
Snapshots: 1

In [11]:
n1.buses.index

Index(['Frankfurt', 'Frankfurt heat', 'Frankfurt gas'], dtype='object', name='Bus')

In [12]:
n1.generators.index

RangeIndex(start=0, stop=0, step=1, name='Generator')

In [13]:
n1.storage_units.index

RangeIndex(start=0, stop=0, step=1, name='StorageUnit')

In [14]:
n1.optimize();

Index(['Frankfurt gas'], dtype='object', name='Store')


Index(['Frankfurt', 'Frankfurt heat', 'Frankfurt gas'], dtype='object', name='Bus')
Index(['OCGT', 'CHP'], dtype='object', name='Link')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.04s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 6 primals, 12 duals
Objective: 1.70e+04
Solver model: available
Solver message: optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Link-ext-p-lower, Link-ext-p-upper, Store-fix-e-lower, Store-fix-e-upper, Store-energy_balance were not assigned to the network.


In [ ]:
n1.loads_t.p

Link,OCGT,CHP
snapshot,,
now,0.0,-3.0


In [19]:
n1.links_t.p0

Link,OCGT,CHP
snapshot,,
now,5.0,10.0


In [18]:
n1.links_t.p1

Link,OCGT,CHP
snapshot,,
now,-2.0,-3.0


In [16]:
n1.links_t.p2

Link,OCGT,CHP
snapshot,,
now,0.0,-3.0


End of the example. 
Starts SMR model versión with network `n2`

In [ ]:
n2 = pypsa.Network()

# Define required carriers first
for carrier in ["electricity", "heat", "nuclear"]:
    n2.add("Carrier", carrier)

# Create buses with defined carriers
n2.add("Bus", "Elec Bus", carrier="electricity")
n2.add("Load", "Electricity Demand", bus="Elec Bus", p_set=5)  # add 5 MW electricity demand

n2.add("Bus", "Heat Bus", carrier="heat") 
n2.add("Load", "Heat Demand", bus="Heat Bus", p_set=3)        # 3 MW heat

n2.add("Bus", "Nuclear Fuel", carrier="nuclear")
# Add uranium storage with nuclear carrier
n2.add("Store", "Uranium Storage",
            bus="Nuclear Fuel",
            carrier="nuclear",  # Explicit carrier assignment
            e_initial=1e6,      # MWh-th equivalent
            e_nom=1e6)

# Define SMR link with proper carrier
n2.add("Link", "SMR_CHP",
    bus0="Nuclear Fuel",
    bus1="Elec Bus",    # Electricity output
    bus2="Heat Bus",    # Heat output
    carrier="nuclear",  # Critical carrier definition
    #p_nom=200,              # 200 MW thermal capacity
    p_nom_extendable=True,
    efficiency=0.35,       # 35% to electricity
    efficiency2=0.45,       # 45% to heat
    capital_cost=8e6        # $8 million/MW
)


n2.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.04s
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: infeasible



('warning', 'infeasible')

In [32]:
print("Electricity Output:", n2.links_t.p1["SMR"].sum())  # -70 MW
print("Heat Output:", n2.links_t.p2["SMR"].sum())         # -108 MW
print("Fuel Consumption:", n2.links_t.p0["SMR"].sum())    # +314.3 MW


KeyError: 'SMR'